# Лабораторна робота №2. Частина 1
**Тема:** Обробка та аналіз даних NOAA

## Завдання 1: Завантаження файлів
Реалізація автоматичного завантаження VHI-індексів для всіх адміністративних областей України. Збереження файлів із міткою часу та перевірка наявності вже завантажених даних для уникнення дублювання.

In [6]:
import os
import urllib.request
import datetime
import pandas as pd

def download_vhi(out_dir="vhi_data"):
    os.makedirs(out_dir, exist_ok=True)
    for i in range(1, 28):
        url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={i}&year1=1981&year2=2024&type=Mean"
        existing_files = [f for f in os.listdir(out_dir) if f.startswith(f"vhi_{i}_")]
        if not existing_files:
            timestamp = datetime.datetime.now().strftime("%Y%m%d%H%M%S")
            filename = f"vhi_{i}_{timestamp}.csv"
            urllib.request.urlretrieve(url, os.path.join(out_dir, filename))

download_vhi()

## Завдання 2: Очищення даних та формування DataFrame
Зчитування текстових файлів, видалення непотрібних HTML-тегів, заповнення пропусків та фільтрація некоректних значень. Заміна ідентифікаторів областей відповідно до української абетки.

In [7]:
def load_and_clean_vhi(data_dir="vhi_data"):
    mapping = {1: 22, 2: 24, 3: 23, 4: 25, 5: 3, 6: 4, 7: 8, 8: 19, 9: 20, 10: 21, 11: 9, 12: 26, 13: 10, 14: 11, 15: 12, 16: 13, 17: 14, 18: 15, 19: 16, 20: 27, 21: 17, 22: 18, 23: 6, 24: 1, 25: 2, 26: 7, 27: 5}
    names = {1: 'Вінницька', 2: 'Волинська', 3: 'Дніпропетровська', 4: 'Донецька', 5: 'Житомирська', 6: 'Закарпатська', 7: 'Запорізька', 8: 'Івано-Франківська', 9: 'Київська', 10: 'Кіровоградська', 11: 'Луганська', 12: 'Львівська', 13: 'Миколаївська', 14: 'Одеська', 15: 'Полтавська', 16: 'Рівненська', 17: 'Сумська', 18: 'Тернопільська', 19: 'Харківська', 20: 'Херсонська', 21: 'Хмельницька', 22: 'Черкаська', 23: 'Чернівецька', 24: 'Чернігівська', 25: 'АР Крим', 26: 'м. Київ', 27: 'м. Севастополь'}
    
    frames = []
    for file in os.listdir(data_dir):
        if file.endswith(".csv"):
            prov_id = int(file.split('_')[1])
            data = []
            with open(os.path.join(data_dir, file), 'r') as f:
                for line in f:
                    line = line.replace('<tt><pre>', '').replace('</pre></tt>', '').strip()
                    if not line or line.startswith('year') or line.startswith('<'):
                        continue
                    parts = [p.strip() for p in line.split(',')]
                    if len(parts) >= 7:
                        data.append(parts[:7])
            
            df = pd.DataFrame(data, columns=['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI'])
            df = df.apply(pd.to_numeric, errors='coerce')
            df = df.dropna()
            df = df[df['VHI'] != -1]
            df['Province_ID'] = mapping[prov_id]
            df['Province_Name'] = names[mapping[prov_id]]
            frames.append(df)
            
    return pd.concat(frames, ignore_index=True)

df_vhi = load_and_clean_vhi()
display(df_vhi.head())

,Year,Week,SMN,SMT,VCI,TCI,VHI,Province_ID,Province_Name
0,1982,1,0.059,258.24,51.11,48.78,49.95,21,Хмельницька
1,1982,2,0.063,261.53,55.89,38.20,47.04,21,Хмельницька
2,1982,3,0.063,263.45,57.30,32.69,44.99,21,Хмельницька
3,1982,4,0.061,265.10,53.96,28.62,41.29,21,Хмельницька
4,1982,5,0.058,266.42,46.87,28.57,37.72,21,Хмельницька


## Пункт 3.1: Фільтрація значень VHI за обраний рік
Створення та тестування функції, яка виділяє часовий ряд показників VHI для конкретного регіону (за його ідентифікатором) протягом вказаного року.

In [10]:
def filter_vhi_by_year(data, region_id, target_year):
    selected_data = data[(data['Province_ID'] == region_id) & (data['Year'] == target_year)]
    return selected_data[['Week', 'VHI']]

print("Результат фільтрації для області 1 за 2020 рік:")
df_year_sample = filter_vhi_by_year(df_vhi, 1, 2020)
display(df_year_sample.head())

Результат фільтрації для області 1 за 2020 рік:


,Week,VHI
34716,1,40.92
34717,2,43.19
34718,3,44.74
34719,4,45.29
34720,5,44.80


## Пункт 3.2: Вибірка даних для групи областей за часовий діапазон
Розробка процедури для отримання підмножини даних індексу VHI, яка охоплює декілька вказаних областей та певний проміжок років (включаючи межі інтервалу).

In [11]:
def get_vhi_multi_regions(data, regions_list, start_year, end_year):
    query_condition = (
        (data['Province_ID'].isin(regions_list)) & 
        (data['Year'] >= start_year) & 
        (data['Year'] <= end_year)
    )
    return data[query_condition][['Year', 'Week', 'Province_Name', 'VHI']]

print("Результат вибірки для областей 9 та 12 за 2018-2020 роки:")
df_range_sample = get_vhi_multi_regions(df_vhi, [9, 12], 2018, 2020)
display(df_range_sample.head())

Результат вибірки для областей 9 та 12 за 2018-2020 роки:


,Year,Week,Province_Name,VHI
4008,2018,1,Київська,40.82
4009,2018,2,Київська,43.65
4010,2018,3,Київська,48.62
4011,2018,4,Київська,51.77
4012,2018,5,Київська,52.08


## Пункт 3.3: Обчислення екстремальних та середніх значень
Реалізація аналітичної функції для визначення ключових статистичних метрик індексу VHI (мінімальне та максимальне значення, середнє арифметичне, медіана) для заданого переліку областей та часового інтервалу.

In [12]:
def calculate_vhi_statistics(data, regions_list, start_year, end_year):
    subset = get_vhi_multi_regions(data, regions_list, start_year, end_year)
    
    metrics = {
        'Мінімум VHI': subset['VHI'].min(),
        'Максимум VHI': subset['VHI'].max(),
        'Середнє значення VHI': subset['VHI'].mean(),
        'Медіана VHI': subset['VHI'].median()
    }
    return metrics

print("Статистичний аналіз для областей 9 та 12 (2018-2020 роки):")
vhi_stats = calculate_vhi_statistics(df_vhi, [9, 12], 2018, 2020)
for label, value in vhi_stats.items():
    print(f"{label}: {value:.2f}")

Статистичний аналіз для областей 9 та 12 (2018-2020 роки):
Мінімум VHI: 20.91
Максимум VHI: 63.62
Середнє значення VHI: 46.78
Медіана VHI: 46.84
